# 01 — Explore and validate

**Depends on:** notebook 00. **Produces:** `schema.json`, `quality_report.json`.

## What this stage is for

Answering three questions about the three datasets before anybody models them:

1. Do they have the same columns, and where they differ, is the difference
   expected?
2. Do shared columns have the same dtype? A count that is `int64` in one and
   `object` in another will union to `object` and then be silently dropped as
   non-numeric by every model. Nobody notices, because the model still trains.
3. What is the null rate per column, and is it plausible?

## The one thing to internalise

**Nulls here are not missing data.** A null count means the event never happened.
A null recency means there is no last-time because there is no first-time. This
is not a data quality problem to be cleaned; it is information, and the
imputation in notebook 02 has to respect it. Filling a never-happened count with
the median of the customers it did happen to invents behaviour that nobody had.
The package encodes this as `DomainImputer` and the whole design is written up in
`trust_score_05/features/docs/feature_library_findings.md` finding 4.5.

In [ ]:
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Every notebook in this track reads and writes here. Nothing else is shared:
# no imports between notebooks, no global state, no kernel that has to stay
# alive. That is what "standalone" means -- you can run notebook 03 tomorrow on
# a different machine as long as this directory came with you.
RUN_DIR = pathlib.Path(os.environ.get("TS05_RUN_DIR", "./_run")).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260601
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print("run directory:", RUN_DIR)
print("contents     :", sorted(p.name for p in RUN_DIR.iterdir()) or "(empty)")

In [ ]:
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert "00" in manifest["stages_complete"], "Run 00-setup-and-data.ipynb first."

DATASETS = ["training_nonfraud", "testing_nonfraud", "fraud"]
frames = {name: pd.read_parquet(manifest["paths"][name]) for name in DATASETS}

for name, frame in frames.items():
    print(f"{name:20s} {frame.shape[0]:7d} rows  {frame.shape[1]:4d} cols")

## The schema

In [ ]:
schema = {
    name: {column: str(dtype) for column, dtype in frame.dtypes.items()}
    for name, frame in frames.items()
}

all_columns = sorted(set().union(*(set(s) for s in schema.values())))
print(f"{len(all_columns)} distinct column name(s) across the three datasets")

rows = []
for column in all_columns:
    present = [name for name in DATASETS if column in schema[name]]
    types = sorted({schema[name][column] for name in present})
    rows.append(
        {
            "column": column,
            "n_present": len(present),
            "present_in": ",".join(present),
            "dtypes": ",".join(types),
            "consistent": len(types) == 1,
        }
    )

overview = pd.DataFrame(rows)
print(f"universal   : {(overview['n_present'] == 3).sum()}")
print(f"partial     : {(overview['n_present'] < 3).sum()}")
print(f"inconsistent: {(~overview['consistent']).sum()}")
overview.head(15)

## Partial columns — expected or not?

The fraud dataset legitimately carries `label`, `case_key` and `fraud_ts`, which
the non-fraud datasets have no reason to have. Anything *else* in this list is a
question for upstream.

In [ ]:
EXPECTED_FRAUD_ONLY = {"label", "case_key", "fraud_ts", "fraud_type"}

partial = overview[overview["n_present"] < 3]
unexpected = partial[~partial["column"].isin(EXPECTED_FRAUD_ONLY)]

print(f"{len(partial)} partial column(s), {len(unexpected)} unexpected")
partial[["column", "present_in", "dtypes"]]

## Dtype disagreements — never benign

If this table has rows, fix the upstream extract before going further. Do not
work around it by casting here, because the cast you choose in a notebook is not
the cast the Spark job will choose.

In [ ]:
inconsistent = overview[~overview["consistent"]]
print(f"{len(inconsistent)} disagreement(s)")
if len(inconsistent):
    display(inconsistent[["column", "present_in", "dtypes"]])
assert len(inconsistent) == 0, "Fix dtype disagreements upstream before modelling."

## Null rates, read as behaviour

Sort descending and read the top of the list as a sentence. `cust_mno_port_days_since_last`
being 94% null does not mean the pipeline is broken; it means 94% of customers
have never ported, which is true and useful. A *count* column being null at all,
on the other hand, is worth a second look — counts should be zero, not null,
unless the entity had no events of that family whatsoever.

In [ ]:
null_rates = pd.DataFrame(
    {name: frame.isna().mean() for name, frame in frames.items()}
).sort_values("training_nonfraud", ascending=False)

print("highest null rates:")
display((null_rates.head(25) * 100).round(2))

print()
print("count columns that are null anywhere (should be rare):")
count_cols = [c for c in null_rates.index if c.endswith("_cnt") or "_cnt_" in c]
suspicious = null_rates.loc[count_cols]
display((suspicious[suspicious["training_nonfraud"] > 0] * 100).round(2))

## Drift between training and testing non-fraud

These two are supposed to be the same population, split at random. If a column's
mean differs materially between them, the split was not random — usually because
it was made on something time-ordered. Worth catching now; a non-random split
makes every evaluation number optimistic.

In [ ]:
numeric = [
    column
    for column in all_columns
    if column in frames["training_nonfraud"]
    and pd.api.types.is_numeric_dtype(frames["training_nonfraud"][column])
    and column not in {"label", "customer_id_numeric"}
]

comparison = pd.DataFrame(
    {
        "train_mean": frames["training_nonfraud"][numeric].mean(),
        "test_mean": frames["testing_nonfraud"][numeric].mean(),
    }
)
comparison["ratio"] = comparison["test_mean"] / comparison["train_mean"].replace(0, np.nan)
comparison["abs_log_ratio"] = np.abs(np.log(comparison["ratio"].replace(0, np.nan)))

print("largest divergence between the two non-fraud splits:")
display(comparison.sort_values("abs_log_ratio", ascending=False).head(12).round(4))

## Fraud versus non-fraud, univariately

This is the diagnostic that tells you whether the problem is hard. Compute a
standardised mean difference per feature. If one feature separates the classes
almost perfectly, either you have found something wonderful or you have found
leakage — and it is almost always leakage.

**Expect `fraud_case_score` to top this list.** It is the planted leak. Seeing it
here, and understanding *why* seeing it here is a red flag rather than a result,
is the entire lesson of this cell.

In [ ]:
non_fraud = frames["training_nonfraud"]
fraud_frame = frames["fraud"]
shared = [c for c in numeric if c in fraud_frame.columns]

effect = []
for column in shared:
    a = non_fraud[column].dropna()
    b = fraud_frame[column].dropna()
    if len(a) < 30 or len(b) < 10:
        continue
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2.0)
    if not np.isfinite(pooled) or pooled == 0:
        continue
    effect.append(
        {
            "feature": column,
            "nonfraud_mean": a.mean(),
            "fraud_mean": b.mean(),
            "std_mean_diff": (b.mean() - a.mean()) / pooled,
        }
    )

effects = pd.DataFrame(effect)
effects["abs_effect"] = effects["std_mean_diff"].abs()
effects = effects.sort_values("abs_effect", ascending=False)

print("strongest univariate separation:")
display(effects.head(20).round(4))

# Why 10 and not 2 or 3. A standardised mean difference of 2-4 is entirely
# ordinary for a genuine fraud feature: fraud really does involve more recent
# SIM changes and more identity lookups, and that is the signal we intend to
# model. A |d| in the tens is a different animal -- it means the column already
# encodes the answer, and no behavioural measurement is that clean. Set this
# threshold to 3 and the detector eats your real features: the models then train
# on the leftovers, score at a lift of about 1.0, and the notebook looks like the
# algorithms failed when in fact selection threw the evidence away. Look at
# `univariate_effects.csv` rather than trusting the constant -- if the printed
# list is more than a handful of columns, the threshold is wrong for your data,
# not the data wrong for the threshold.
LEAK_EFFECT_THRESHOLD = 10.0

leaks = effects[effects["abs_effect"] > LEAK_EFFECT_THRESHOLD]["feature"].tolist()
print()
print(f"features with implausibly strong separation (|d| > {LEAK_EFFECT_THRESHOLD:.0f})"
      " -- suspect leakage:")
print(leaks)
print()
print("for contrast, the strongest features NOT flagged (these should look like"
      " plausible fraud behaviour, not like the answer):")
print(effects[effects["abs_effect"] <= LEAK_EFFECT_THRESHOLD].head(5)["feature"].tolist())

## Build the quality report

Each check gets a severity. `error` means stop. This mirrors the `Check` /
`DataQualityReport` pair in `trust_score_05.ml.discovery`, deliberately, so that
moving between the two tracks is not a relearning exercise.

In [ ]:
checks = []


def check(name, severity, message, **details):
    checks.append(
        {"name": name, "severity": severity, "message": message, "details": details}
    )


check(
    "dataset_row_counts",
    "info",
    "row count per dataset",
    **{name: int(len(frame)) for name, frame in frames.items()},
)

if len(inconsistent):
    check(
        "dtype_disagreement",
        "error",
        f"{len(inconsistent)} column(s) have different dtypes across datasets",
        columns=inconsistent["column"].tolist(),
    )
else:
    check("dtype_disagreement", "info", "all shared columns agree on dtype")

if len(unexpected):
    check(
        "unexpected_partial_columns",
        "warning",
        f"{len(unexpected)} column(s) are missing from some datasets without an "
        "expected reason",
        columns=unexpected["column"].tolist(),
    )

very_null = null_rates[null_rates["training_nonfraud"] > 0.98]
if len(very_null):
    check(
        "near_empty_columns",
        "warning",
        f"{len(very_null)} column(s) are more than 98% null in training",
        columns=very_null.index.tolist(),
    )

constant = [c for c in numeric if frames["training_nonfraud"][c].nunique(dropna=True) <= 1]
if constant:
    check(
        "constant_columns",
        "warning",
        f"{len(constant)} column(s) have no variance and cannot carry signal",
        columns=constant,
    )

if leaks:
    check(
        "suspected_label_leakage",
        "error",
        f"{len(leaks)} column(s) separate fraud from non-fraud implausibly well",
        columns=leaks,
    )

if len(fraud_frame) == 0:
    check("empty_fraud_population", "error", "the fraud dataset has no rows")

report = {
    "notebook": "01-explore-and-validate",
    "checks": checks,
    "severity": (
        "error" if any(c["severity"] == "error" for c in checks)
        else "warning" if any(c["severity"] == "warning" for c in checks)
        else "info"
    ),
}

for entry in sorted(checks, key=lambda c: {"error": 0, "warning": 1, "info": 2}[c["severity"]]):
    print(f"[{entry['severity']:7s}] {entry['name']}: {entry['message']}")
    for key, value in entry["details"].items():
        shown = value[:8] if isinstance(value, list) else value
        print(f"          {key}: {shown}")
print()
print("worst severity:", report["severity"])

## Write it down and move on

**Note that this stage ends with a deliberate `error` severity** when run on the
synthetic data, because of the planted leak. That is correct. Notebook 02 is
where the leak gets excluded, and it reads `suspected_label_leakage` from this
report to do it — so the check is not just a warning, it is an input.

In [ ]:
(RUN_DIR / "schema.json").write_text(json.dumps(schema, indent=2))
(RUN_DIR / "quality_report.json").write_text(json.dumps(report, indent=2, default=str))
null_rates.to_csv(RUN_DIR / "null_rates.csv")
effects.to_csv(RUN_DIR / "univariate_effects.csv", index=False)

manifest["stages_complete"] = sorted(set(manifest["stages_complete"]) | {"01"})
manifest["paths"].update(
    {
        "schema": str(RUN_DIR / "schema.json"),
        "quality_report": str(RUN_DIR / "quality_report.json"),
        "null_rates": str(RUN_DIR / "null_rates.csv"),
        "univariate_effects": str(RUN_DIR / "univariate_effects.csv"),
    }
)
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("wrote schema.json, quality_report.json, null_rates.csv, univariate_effects.csv")
print("Stage 01 complete. Open 02-select-features.ipynb.")